# Diebold-Li NL 模型（Jupyter 版）

该 notebook 将原 Stata do-file `nl_did_vpyield_fixed.do` 的流程完全迁移到 Python，便于逐步调试：

1. 读取债券收益率 CSV，并完成日期/数值字段转换。
2. 构造 Treated/Post/DiD 指示变量与 BdType 过滤、时间窗口筛样。
3. 在 SciPy `least_squares` 中嵌入交替固定效应（公司、月份）剥离，估计 Diebold-Li 型收益率曲线。
4. 输出系数表、预测值、散点图、边际效应以及 `.parquet`/`.dta` 数据快照。
5. **[新增]** 平行趋势检验（事件研究法）与安慰剂检验。

可根据需要修改参数单元中的路径、事件日期、窗口等配置，然后自上而下依次执行。


## 固定效应剥离算法说明

我们估计的非线性模型在每一次参数更新后，都需要从残差中剥离公司固定效应 $\alpha_i$ 与月份固定效应 $\gamma_t$，实质上是在求解
$$y_{it} = f_{it}(\theta) + \alpha_i + \gamma_t + \varepsilon_{it},$$
其中 $f_{it}(\theta)$ 为 Diebold-Li 结构项（含 DiD 交互）。令 $r_{it}=y_{it}-f_{it}(\theta)$，在给定当前 $\alpha_i^{(k)}$ 与 $\gamma_t^{(k)}$ 的情况下，通过交替投影（Gauss-Seidel）更新：

1. **更新月份效应**（先减去上一轮的公司效应）
$$\gamma_t^{(k+1)} = \frac{1}{|I_t|} \sum_{i \in I_t} \big(r_{it} - \alpha_i^{(k)}\big),$$
其中 $I_t$ 表示在月份 $t$ 内观测到的公司集合。
2. **更新公司效应**（再减去最新月份效应）
$$\alpha_i^{(k+1)} = \frac{1}{|T_i|} \sum_{t \in T_i} \big(r_{it} - \gamma_t^{(k+1)}\big),$$
其中 $T_i$ 为公司 $i$ 的观察月份集合。
3. **中心化约束**：为避免不可识别的常数漂移，使用
$$c^{(k+1)} = \frac{\bar{\alpha}^{(k+1)} + \bar{\gamma}^{(k+1)}}{2},$$
并更新 $\alpha_i^{(k+1)} \leftarrow \alpha_i^{(k+1)} - c^{(k+1)}$，$\gamma_t^{(k+1)} \leftarrow \gamma_t^{(k+1)} - c^{(k+1)}$。

重复以上步骤直至 $\max_{i,t} |(\alpha_i^{(k+1)}+\gamma_t^{(k+1)}) - (\alpha_i^{(k)}+\gamma_t^{(k)})| < \text{tol}$。这套交替均值消除方法与 (Abowd 等, 1999) 提出的高维固定效应处理、以及 (Guimarães & Portugal, 2010) 针对 Stata 两维 FE 的可行算法一致，可在复杂非线性框架下保持与传统“within”变换相同的剥离效果。

**参考文献**  
[1] Guimarães P, Portugal P. A simple feasible procedure to fit models with
high-dimensional fixed effects. *The Stata Journal*, 2010, 10(4): 628-649. https://journals.sagepub.com/doi/pdf/10.1177/1536867X1101000406

In [ ]:
from __future__ import annotations

import json
import logging
from pathlib import Path
from typing import Dict, Iterable, List, Tuple
import argparse

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from numpy.typing import NDArray
from scipy.optimize import least_squares
from scipy.stats import t as student_t

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logging.info("Libraries imported and logging configured.")

In [ ]:
# === 参数设置（可根据需要修改） ===
INPUT_CSV = Path("./bond_return_with_kcbz.csv")
OUTPUT_DIR = Path("./outputs_notebook")
KEY_DATE = pd.Timestamp("2025-06-18")
WINDOW_DAYS = 90
MAX_FE_ITERS = 50
FE_TOL = 1e-10
MAX_NL_STEPS = 10_000
INITIAL_L = 1.0
SCATTER_FILENAME = "scatter_by_bdtype.png"
EXPORT_DTA = True

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
logging.info("Parameters configured. Output directory: %s", OUTPUT_DIR.resolve())

In [ ]:
def read_csv(path: Path) -> pd.DataFrame:
    """读取原始 CSV 并输出形状，用于快速确认输入数据量。"""
    logging.info("Reading CSV: %s", path)
    df = pd.read_csv(path)
    logging.debug("Raw shape: %s", df.shape)
    return df


def coerce_date_column(df: pd.DataFrame, column: str) -> None:
    """把指定列转换为日期；缺失列时仅提示，不终止流程。"""
    if column not in df.columns:
        logging.info("Date column `%s` not found; skipping.", column)
        return
    df[column] = pd.to_datetime(df[column], errors="coerce")
    logging.debug(
        "Converted `%s` to datetime. Missing ratio: %.3f",
        column,
        df[column].isna().mean(),
    )


def coerce_numeric_column(df: pd.DataFrame, column: str) -> None:
    """将字符串数字安全转换为浮点，便于后续矩阵计算。"""
    if column not in df.columns:
        logging.warning("Numeric column `%s` not found.", column)
        return
    df[column] = pd.to_numeric(df[column], errors="coerce")
    logging.debug(
        "Converted `%s` to numeric. Missing ratio: %.3f",
        column,
        df[column].isna().mean(),
    )


def normalize_kcbz(df: pd.DataFrame) -> None:
    """将 kcbz 字段统一转换为 0/1 dummy，并创建 Treated 变量。"""
    col = "kcbz"
    if col not in df.columns:
        logging.error("`kcbz` not found; defaulting Treated to 0.")
        df[col] = 0

    if pd.api.types.is_numeric_dtype(df[col]):
        numeric = df[col]
    else:
        lowered = df[col].astype(str).str.strip().str.lower()
        mapping = {
            "1": 1,
            "y": 1,
            "yes": 1,
            "true": 1,
            "t": 1,
            "0": 0,
            "n": 0,
            "no": 0,
            "false": 0,
            "f": 0,
        }
        numeric = lowered.map(mapping)
        numeric = numeric.fillna(pd.to_numeric(df[col], errors="coerce"))
        numeric = numeric.fillna(0)

    df[col] = (numeric.fillna(0) != 0).astype(np.int8)
    df["Treated"] = df[col].astype(np.int8)
    logging.debug("Created Treated dummy from `kcbz`.")


def ensure_end_date(df: pd.DataFrame) -> None:
    """确保存在 EndDt 列，否则回退到 ValueDt 或 IssDt。"""
    if "EndDt" in df.columns:
        return
    for candidate in ("ValueDt", "IssDt"):
        if candidate in df.columns:
            df["EndDt"] = pd.to_datetime(df[candidate], errors="coerce")
            logging.info("Using `%s` as EndDt.", candidate)
            return
    raise ValueError("No EndDt/ValueDt/IssDt found. Cannot proceed.")


def filter_bdtype(df: pd.DataFrame) -> pd.DataFrame:
    """筛选 BdType ∈ {13,14,19}，与 Stata keep 逻辑一致。"""
    if "BdType" not in df.columns:
        raise ValueError("BdType not found. Required for filtering.")
    numeric = pd.to_numeric(df["BdType"], errors="coerce")
    mask = numeric.isin({13, 14, 19})
    kept = df.loc[mask].copy()
    logging.info(
        "Filtered BdType. Kept %d / %d rows (%.1f%%).",
        kept.shape[0],
        df.shape[0],
        100 * kept.shape[0] / max(df.shape[0], 1),
    )
    kept["BdType"] = numeric.loc[mask].astype("Int64")
    return kept


def assign_time_window(
    df: pd.DataFrame, key_date: pd.Timestamp, window_days: int
) -> pd.DataFrame:
    """生成时间窗口样本、Post、DiD 变量，完全对齐 Stata 中的逻辑。"""
    if "EndDt" not in df.columns:
        raise ValueError("EndDt is required before time window filtering.")

    df = df.copy()
    df["time_window"] = df["EndDt"].sub(key_date).dt.days.abs() <= window_days
    before = df.shape[0]
    df = df.loc[df["time_window"]].copy()
    logging.info(
        "Applied ±%d day window around %s. Kept %d / %d rows.",
        window_days,
        key_date.date(),
        df.shape[0],
        before,
    )
    df["Post"] = (df["EndDt"] >= key_date).astype(np.int8)
    df["DiD"] = (df["Treated"] * df["Post"]).astype(np.int8)
    return df


def ensure_company_code(df: pd.DataFrame) -> None:
    """确认 CompanyCode 存在并转为字符串，避免 merge/panel 歧义。"""
    if "CompanyCode" not in df.columns:
        raise ValueError("CompanyCode not found; needed for firm FE.")
    df["CompanyCode"] = df["CompanyCode"].astype(str)


def prepare_dataframe(
    csv_path: Path, key_date: pd.Timestamp, window_days: int
) -> pd.DataFrame:
    """按 Stata 步骤顺序完成所有清洗、筛选、指示变量生成。"""
    df = read_csv(csv_path)

    # 1) 日期列：尝试转换 EndDt/ValueDt/MatDt
    for dcol in ("EndDt", "ValueDt", "MatDt"):
        if dcol in df.columns:
            coerce_date_column(df, dcol)
    ensure_end_date(df)
    df["EndDt_month"] = df["EndDt"].dt.year * 100 + df["EndDt"].dt.month

    # 2) 数值列：收益率、TrueYield、久期
    for vcol in ("VPYield", "Maturity", "TrueYield"):
        coerce_numeric_column(df, vcol)

    # 3) 处理组 dummy、公司识别
    normalize_kcbz(df)
    ensure_company_code(df)
    # 4) 债券类型筛选 + 时间窗口 + 有效观测
    df = filter_bdtype(df)
    df = assign_time_window(df, key_date, window_days)
    df = df[df["Maturity"].notna() & df["Maturity"].gt(0)]
    df = df[df["VPYield"].notna()]
    df = df[df["VPYield"] <= 20]
    df = df[df["EndDt_month"].notna()]

    # Drop rows with missing inputs used by the evaluator.
    required = [
        "VPYield",
        "Maturity",
        "Post",
        "Treated",
        "DiD",
        "CompanyCode",
        "EndDt_month",
    ]
    before = df.shape[0]
    df = df.dropna(subset=required)
    logging.info(
        "Dropped %d rows due to missing required fields.", before - df.shape[0]
    )
    df.reset_index(drop=True, inplace=True)
    return df


def describe_data(df: pd.DataFrame) -> None:
    """输出关键变量的描述统计，便于和 Stata summarize 对照。"""
    cols = ["VPYield", "Maturity", "Post", "Treated", "DiD"]
    available = [c for c in cols if c in df.columns]
    if not available:
        return
    summary = df[available].describe().transpose()
    logging.info("Descriptive statistics:\n%s", summary.to_string(float_format="%.4f"))

In [ ]:
PARAM_ORDER: List[str] = [
    "b1Cb",
    "b2Cb",
    "b3Cb",
    "b1Ca",
    "b2Ca",
    "b3Ca",
    "b1Tb",
    "b2Tb",
    "b3Tb",
    "b1Ta",
    "b2Ta",
    "b3Ta",
    "L",
]


def safe_to_parquet(df: pd.DataFrame, path: Path) -> Path:
    """
    优先写入 Parquet；若环境缺少 pyarrow/fastparquet，则降级为 CSV 并记录警告。
    返回实际写入的文件路径，便于日志提示。
    """
    try:
        df.to_parquet(path, index=False)
        return path
    except (ImportError, ModuleNotFoundError) as exc:
        fallback_path = path.with_suffix(".csv")
        logging.warning(
            "Parquet engine unavailable (%s); falling back to CSV: %s",
            exc,
            fallback_path,
        )
        df.to_csv(fallback_path, index=False)
        return fallback_path


def positive_float(value: str) -> float:
    """确保 CLI 输入的浮点数为正数，避免无效的 L 初值或窗口设置。"""
    parsed = float(value)
    if parsed <= 0:
        raise argparse.ArgumentTypeError("Expected a positive float.")
    return parsed


def _group_mean(
    values: NDArray[np.float64], groups: NDArray[np.int64], n_groups: int
) -> NDArray[np.float64]:
    """使用 np.bincount 计算分组均值，复刻 bysort egen mean 的效果。"""
    sums = np.bincount(groups, weights=values, minlength=n_groups)
    counts = np.bincount(groups, minlength=n_groups)
    means = np.divide(
        sums,
        counts,
        out=np.zeros_like(sums, dtype=np.float64),
        where=counts > 0,
    )
    return means


def alternating_fe(
    resid: NDArray[np.float64],
    company_idx: NDArray[np.int64],
    month_idx: NDArray[np.int64],
    n_companies: int,
    n_months: int,
    *,
    maxiter: int,
    tol: float,
) -> Tuple[NDArray[np.float64], NDArray[np.float64], int]:
    """迭代在公司/月份之间交替平滑固定效应，直到收敛或达到上限。"""
    alpha = np.zeros(n_companies, dtype=np.float64)
    gamma = np.zeros(n_months, dtype=np.float64)

    for iteration in range(1, maxiter + 1):
        alpha_old = alpha.copy()
        gamma_old = gamma.copy()

        # Step A: 固定公司效应，利用月份均值更新月份效应 gamma
        tmp = resid - alpha[company_idx]
        gamma = _group_mean(tmp, month_idx, n_months)

        # Step B: 固定最新月份效应，按公司均值更新 alpha
        tmp2 = resid - gamma[month_idx]
        alpha = _group_mean(tmp2, company_idx, n_companies)

        center = alpha.mean() + gamma.mean()
        alpha -= center / 2
        gamma -= center / 2

        combined_new = alpha[company_idx] + gamma[month_idx]
        combined_old = alpha_old[company_idx] + gamma_old[month_idx]
        maxchg = np.max(np.abs(combined_new - combined_old))
        if maxchg < tol:
            logging.debug("FE converged in %d iterations.", iteration)
            return alpha, gamma, iteration

    logging.warning(
        "FE solver hit max iterations (%d) without convergence (tol %.1e).",
        maxiter,
        tol,
    )
    return alpha, gamma, maxiter


class DieboldLiFEModel:
    """封装 Diebold-Li 结构项与公司/月度固定效应求解，便于 SciPy 调用。"""

    def __init__(
        self,
        df: pd.DataFrame,
        *,
        max_fe_iters: int,
        fe_tol: float,
    ) -> None:
        """准备模型所需的 numpy 缓存和索引，加速后续迭代。"""
        self.df = df.copy()
        self.y = df["VPYield"].to_numpy(dtype=np.float64)
        self.maturity = df["Maturity"].to_numpy(dtype=np.float64)
        self.post = df["Post"].to_numpy(dtype=np.float64)
        self.treated = df["Treated"].to_numpy(dtype=np.float64)
        self.did = df["DiD"].to_numpy(dtype=np.float64)

        companies, company_idx = np.unique(
            df["CompanyCode"].astype(str), return_inverse=True
        )
        months, month_idx = np.unique(
            df["EndDt_month"].astype(int), return_inverse=True
        )

        self.company_codes = companies
        self.month_codes = months
        self.company_idx = company_idx.astype(np.int64)
        self.month_idx = month_idx.astype(np.int64)
        self.n_companies = companies.shape[0]
        self.n_months = months.shape[0]

        self.max_fe_iters = max_fe_iters
        self.fe_tol = fe_tol
        self.latest_alpha = np.zeros(self.n_companies)
        self.latest_gamma = np.zeros(self.n_months)
        self.latest_terms: Dict[str, NDArray[np.float64]] = {}
        self.last_fe_iterations = 0

    @staticmethod
    def _terms(
        maturity: NDArray[np.float64], L: float
    ) -> Tuple[NDArray[np.float64], NDArray[np.float64]]:
        """计算 Diebold-Li 中的两个基函数 term1/term2，并对 L→0 做数值保护。"""
        safe_L = np.clip(L, 1e-10, None)
        denom = safe_L * maturity
        term1 = np.ones_like(maturity)
        mask = (maturity > 0) & (safe_L > 0)
        term1[mask] = (1 - np.exp(-safe_L * maturity[mask])) / denom[mask]
        term2 = term1 - np.exp(-safe_L * maturity)
        return term1, term2

    def structural_component(
        self, params: NDArray[np.float64]
    ) -> Tuple[NDArray[np.float64], NDArray[np.float64], NDArray[np.float64]]:
        """根据参数向量生成结构部分拟合值，并缓存 term1/term2。"""
        (
            b1Cb,
            b2Cb,
            b3Cb,
            b1Ca,
            b2Ca,
            b3Ca,
            b1Tb,
            b2Tb,
            b3Tb,
            b1Ta,
            b2Ta,
            b3Ta,
            L,
        ) = params

        term1, term2 = self._terms(self.maturity, L)

        # 结构项：依次包含基准载荷（Cb）、Post 载荷（Ca）、Treat 载荷（Tb）、DiD 载荷（Ta）
        fitted = (
            b1Cb
            + b2Cb * term1
            + b3Cb * term2
            + b1Ca * self.post
            + b2Ca * self.post * term1
            + b3Ca * self.post * term2
            + b1Tb * self.treated
            + b2Tb * self.treated * term1
            + b3Tb * self.treated * term2
            + b1Ta * self.did
            + b2Ta * self.did * term1
            + b3Ta * self.did * term2
        )

        self.latest_terms = {"term1": term1, "term2": term2}
        return fitted, term1, term2

    def residuals(self, params: NDArray[np.float64]) -> NDArray[np.float64]:
        """供 least_squares 调用：先算结构项，再交替剥离固定效应，返回残差。"""
        fitted, _, _ = self.structural_component(params)
        resid = self.y - fitted
        alpha, gamma, iters = alternating_fe(
            resid,
            self.company_idx,
            self.month_idx,
            self.n_companies,
            self.n_months,
            maxiter=self.max_fe_iters,
            tol=self.fe_tol,
        )
        self.latest_alpha = alpha
        self.latest_gamma = gamma
        self.last_fe_iterations = iters
        fe_adjustment = alpha[self.company_idx] + gamma[self.month_idx]
        final_resid = self.y - (fitted + fe_adjustment)
        return final_resid

    def predict(self, params: NDArray[np.float64]) -> NDArray[np.float64]:
        """在估计完成后返回含 FE 的总体预测值，用于作图/导出。"""
        fitted, _, _ = self.structural_component(params)
        alpha_obs = self.latest_alpha[self.company_idx]
        gamma_obs = self.latest_gamma[self.month_idx]
        return fitted + alpha_obs + gamma_obs

    def margins_did(
        self,
        params: NDArray[np.float64],
        maturities: Iterable[float],
        cov: NDArray[np.float64],
        dof: int,
    ) -> pd.DataFrame:
        """计算不同期限下 DiD 的边际效应，附带 t 检验与 p 值，并在控制台打印。"""
        maturities = np.asarray(list(maturities), dtype=np.float64)
        term1, term2 = self._terms(maturities, params[-1])

        idx = [PARAM_ORDER.index(name) for name in ("b1Ta", "b2Ta", "b3Ta")]
        beta = params[idx]
        cov_sub = cov[np.ix_(idx, idx)]

        weights = np.column_stack((np.ones_like(maturities), term1, term2))
        effects = weights @ beta
        variances = np.einsum("ij,jk,ik->i", weights, cov_sub, weights)
        std_err = np.sqrt(np.maximum(variances, 0))
        with np.errstate(divide="ignore", invalid="ignore"):
            t_values = np.divide(
                effects, std_err, out=np.zeros_like(effects), where=std_err > 0
            )
        p_values = 2 * student_t.sf(np.abs(t_values), dof)

        crit = student_t.ppf(0.975, dof)
        ci_low = effects - crit * std_err
        ci_high = effects + crit * std_err

        margins_df = pd.DataFrame(
            {
                "Maturity": maturities,
                "term1": term1,
                "term2": term2,
                "dydx_DiD": effects,
                "std_err": std_err,
                "t_value": t_values,
                "p_value": p_values,
                "ci_low": ci_low,
                "ci_high": ci_high,
            }
        )

        print("\nDiD marginal effects with t-tests")
        print("-" * 86)
        print(
            f"{'Maturity':>10} | {'Effect':>10}  {'Std. Err.':>10}  {'t':>7}  {'P>|t|':>7}     [95% conf. interval]"
        )
        print("-" * 86)
        for row in margins_df.itertuples(index=False):
            print(
                f"{row.Maturity:10.2f} | {row.dydx_DiD:10.6f}  {row.std_err:10.6f}  "
                f"{row.t_value:7.2f}  {row.p_value:7.3f}     {row.ci_low:10.6f}  {row.ci_high:10.6f}"
            )
        print("-" * 86)

        return margins_df


def run_nl_regression(
    model: DieboldLiFEModel,
    *,
    initial_L: float,
    max_nl_steps: int,
    verbose: bool,
) -> Tuple[np.ndarray, least_squares, List[float]]:
    """调用 SciPy 最小二乘以 Stata 的初值为起点估计全部参数。"""
    # 与 Stata initial() 一致的初值（除 L 由 CLI 控制）
    initial_guess = np.array(
        [
            0.01,  # b1Cb
            0.01,  # b2Cb
            0.01,  # b3Cb
            0.0,  # b1Ca
            0.0,  # b2Ca
            0.0,  # b3Ca
            0.0,  # b1Tb
            0.0,  # b2Tb
            0.0,  # b3Tb
            0.0,  # b1Ta
            0.0,  # b2Ta
            0.0,  # b3Ta
            initial_L,  # L
        ],
        dtype=np.float64,
    )

    # L 需要保持正值，其他参数允许正负
    lower_bounds = np.array([-np.inf] * (len(PARAM_ORDER) - 1) + [1e-10])
    upper_bounds = np.array([np.inf] * len(PARAM_ORDER))

    # 记录每次迭代的残差平方和 (SSE)
    iteration_history: List[float] = []
    initial_resid = model.residuals(initial_guess)
    iteration_history.append(float(np.sum(initial_resid**2)))

    def _callback(*args) -> None:
        """兼容不同 SciPy 版本的回调接口，记录 SSE 轨迹。"""
        if len(args) == 3:
            _, cost, _ = args
            sse = float(2 * cost)
        else:
            params_current = args[0]
            resid = model.residuals(params_current)
            sse = float(np.sum(resid**2))
        iteration_history.append(sse)

    lsq_kwargs = dict(
        fun=model.residuals,
        x0=initial_guess,
        bounds=(lower_bounds, upper_bounds),
        max_nfev=max_nl_steps,
        verbose=2 if verbose else 0,
    )

    try:
        lsq = least_squares(**lsq_kwargs, callback=_callback)
    except TypeError as exc:
        if "callback" not in str(exc):
            raise
        logging.info(
            "SciPy least_squares callback unsupported; falling back without intermediate tracking."
        )
        lsq = least_squares(**lsq_kwargs)
        iteration_history.append(float(2 * lsq.cost))
    if not lsq.success:
        logging.warning("Least squares did not converge: %s", lsq.message)

    params = lsq.x
    return params, lsq, iteration_history


def summarize_results(
    model: DieboldLiFEModel, params: NDArray[np.float64], lsq: least_squares
) -> Tuple[pd.DataFrame, Dict[str, float], NDArray[np.float64]]:
    """计算 SSE/R²/标准误并生成结果表，方便与 Stata estimates table 对照。"""
    residuals = lsq.fun
    n_obs = residuals.size
    dof = max(n_obs - len(params) - model.n_companies + model.n_months + 1, 1)
    sse = np.sum(residuals**2)
    mse = sse / dof
    y = model.y
    sst = np.sum((y - np.mean(y)) ** 2)
    r2 = 1 - sse / sst if sst > 0 else np.nan

    jac = lsq.jac
    try:
        cov = mse * np.linalg.inv(jac.T @ jac)
    except np.linalg.LinAlgError:
        cov = mse * np.linalg.pinv(jac.T @ jac)

    stderr = np.sqrt(np.diag(cov))
    t_values = params / stderr
    p_values = 2 * student_t.sf(np.abs(t_values), dof)
    crit = student_t.ppf(0.975, dof)
    ci_low = params - crit * stderr
    ci_high = params + crit * stderr
    results = pd.DataFrame(
        {
            "param": PARAM_ORDER,
            "estimate": params,
            "std_err": stderr,
            "t_value": t_values,
            "p_value": p_values,
            "ci_low": ci_low,
            "ci_high": ci_high,
        }
    )
    adj_r2 = 1 - (1 - r2) * (n_obs - 1) / max(n_obs - len(params), 1)
    stats = {
        "r2": r2,
        "adj_r2": adj_r2,
        "root_mse": np.sqrt(mse),
        "res_dev": -2 * lsq.cost,
        "n_obs": n_obs,
        "sse": sse,
        "mse": mse,
        "dof": dof,
    }
    return results, stats, cov


def save_outputs(
    df: pd.DataFrame,
    params_df: pd.DataFrame,
    margins_df: pd.DataFrame,
    *,
    output_dir: Path,
    scatter_filename: str,
    export_dta: bool,
) -> None:
    """将估计输出写入 CSV/Parquet/图像/可选 Stata dta，复用输出目录。"""
    output_dir.mkdir(parents=True, exist_ok=True)

    params_path = output_dir / "nl_regression_results.csv"
    params_df.to_csv(params_path, index=False)
    logging.info("Saved parameter table to %s", params_path)

    margins_path = output_dir / "margins_did.csv"
    margins_df.to_csv(margins_path, index=False)
    logging.info("Saved margins table to %s", margins_path)

    data_path = output_dir / "did_analysis_data2.parquet"
    actual_data_path = safe_to_parquet(df, data_path)
    logging.info("Saved processed dataset to %s", actual_data_path)

    if export_dta:
        dta_path = output_dir / "did_analysis_data2.dta"
        df.to_stata(dta_path, write_index=False, version=118)
        logging.info("Saved processed dataset to %s", dta_path)

    if "y_hat" in df.columns:
        plt.figure(figsize=(8, 6))
        sns.scatterplot(data=df, x="y_hat", y="VPYield", hue="BdType", palette="deep")
        plt.title("Actual vs Predicted VPYield (colored by BdType)")
        plt.xlabel("Predicted VPYield")
        plt.ylabel("Actual VPYield")
        plt.tight_layout()
        scatter_path = output_dir / scatter_filename
        plt.savefig(scatter_path, dpi=300)
        plt.close()
        logging.info("Saved scatter plot to %s", scatter_path)


def print_iteration_history(history: List[float]) -> None:
    """按照指定格式打印每次迭代的残差平方和。"""
    print()
    for idx, value in enumerate(history):
        label = f"Iteration {idx}:"
        print(f"{label:<12} Residual SS = {value:11.5f}")
    print()


def print_regression_summary(stats: Dict[str, float], results: pd.DataFrame) -> None:
    """以 Stata 风格输出回归概览与系数表。"""
    print(
        f"Nonlinear regression".ljust(52) + f"Number of obs = {stats['n_obs']:>10,.0f}"
    )
    print("".ljust(52) + f"DOF           = {stats['dof']:>10,.0f}")
    print("".ljust(52) + f"R-squared     = {stats['r2']:>10.4f}")
    print("".ljust(52) + f"Adj R-squared = {stats['adj_r2']:>10.4f}")
    print("".ljust(52) + f"Root MSE      = {stats['root_mse']:>10.6f}")
    """print("".ljust(52) + f"Res. dev.     = {stats['res_dev']:>10.3f}")"""

    print("\n" + "-" * 86)
    print(
        f"{'Param':>12} | {'Coefficient':>11}  {'Std. Err.':>10}  {'t':>7}  {'P>|t|':>7}     [95% conf. interval]"
    )
    print("-" * 86)
    for row in results.itertuples(index=False):
        name = f"/{row.param}"
        print(
            f"{name:>12} | {row.estimate:11.6f}  {row.std_err:10.6f}  "
            f"{row.t_value:7.2f}  {row.p_value:7.3f}     {row.ci_low:10.6f}  {row.ci_high:10.6f}"
        )
    print("-" * 86 + "\n")

In [ ]:
# === 平行趋势检验与安慰剂检验扩展模块 ===

class DieboldLiEventStudyModel(DieboldLiFEModel):
    """
    扩展模型以支持事件研究（Event Study），用于平行趋势检验。
    将 DiD 交互项替换为 Treated * RelativeMonth 的一系列交互。
    """
    def __init__(
        self, 
        df: pd.DataFrame, 
        key_date: pd.Timestamp,
        window_months: int = 3,
        **kwargs
    ):
        super().__init__(df, **kwargs)
        self.key_date = key_date
        self.window_months = window_months
        
        # 计算相对月份：(Year - KeyYear) * 12 + (Month - KeyMonth)
        key_y, key_m = key_date.year, key_date.month
        # df['EndDt_month'] 格式为 yyyymm
        y = (df['EndDt_month'] // 100).astype(int)
        m = (df['EndDt_month'] % 100).astype(int)
        
        self.rel_month = (y - key_y) * 12 + (m - key_m)
        
        # 生成相对时间 dummy 列表 [-window, ..., +window]
        # 选择 -1 作为基准期（归零），不包含在回归项中
        self.rel_periods = [t for t in range(-window_months, window_months + 1) if t != -1]
        
        # 预计算每个相对期的 dummy * Treated
        self.event_dummies = {}
        for t in self.rel_periods:
            mask = (self.rel_month == t) & (self.treated == 1)
            self.event_dummies[t] = mask.astype(np.float64)
            
        # 参数说明：
        # Base(9): Cb(3) + Ca(3) + Tb(3)
        # Event(3 * n_periods): 每个相对期都有 Level, Slope, Curvature 三个系数
        # L(1): 衰减因子
        self.base_params_count = 9 
        
    def structural_component(
        self, params: np.ndarray
    ) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        # 解包基础参数
        base_params = params[:9]
        event_params = params[9:-1]
        L = params[-1]
        
        b1Cb, b2Cb, b3Cb = base_params[0:3]
        b1Ca, b2Ca, b3Ca = base_params[3:6]
        b1Tb, b2Tb, b3Tb = base_params[6:9]
        
        term1, term2 = self._terms(self.maturity, L)
        
        # 基础部分：保留 Cb(常数), Ca(时间趋势), Tb(组别差异)，去掉了单一的 DiD 项
        fitted = (
            b1Cb + b2Cb * term1 + b3Cb * term2 +
            b1Ca * self.post + b2Ca * self.post * term1 + b3Ca * self.post * term2 +
            b1Tb * self.treated + b2Tb * self.treated * term1 + b3Tb * self.treated * term2
        )
        
        # 事件研究部分：叠加每一期的动态效应
        for i, t in enumerate(self.rel_periods):
            p_idx = i * 3
            b1_t = event_params[p_idx]
            b2_t = event_params[p_idx+1]
            b3_t = event_params[p_idx+2]
            
            dummy = self.event_dummies[t]
            fitted += (
                b1_t * dummy +
                b2_t * dummy * term1 +
                b3_t * dummy * term2
            )
            
        return fitted, term1, term2

def run_parallel_trend_test(
    df: pd.DataFrame,
    key_date: pd.Timestamp,
    fixed_L: float,
    window_months: int = 4
):
    """
    执行平行趋势检验。为保证稳定性，建议固定 L 为主回归的估计值。
    """
    print(f"Running Event Study (Window: +/-{window_months} months)...")
    
    # 实例化事件研究模型
    # 注意：这里我们可以稍微放宽 FE 收敛要求以加快速度
    model = DieboldLiEventStudyModel(
        df, key_date, window_months=window_months, 
        max_fe_iters=30, fe_tol=1e-8
    )
    
    # 构造初始参数
    n_event_params = len(model.rel_periods) * 3
    n_total = 9 + n_event_params + 1
    
    initial_guess = np.zeros(n_total)
    initial_guess[0] = 2.0  # Constant level approx
    initial_guess[-1] = fixed_L 
    
    # 锁定 L (通过极窄的上下界)
    lower = [-np.inf] * (n_total - 1) + [fixed_L - 1e-8]
    upper = [np.inf] * (n_total - 1) + [fixed_L + 1e-8]
    
    res = least_squares(
        model.residuals,
        initial_guess,
        bounds=(lower, upper),
        max_nfev=500, # 主要是线性参数，收敛较快
        verbose=1
    )
    
    # 提取 Level Factor (b1) 的系数进行绘图
    # 如果您关心斜率或曲率，可以修改 idx 取 p_idx+1 或 p_idx+2
    results_list = []
    
    # 计算简易标准误 (基于 Jacobian)
    mse = (res.fun ** 2).sum() / (len(res.fun) - n_total)
    try:
        cov = mse * np.linalg.inv(res.jac.T @ res.jac)
        stderr = np.sqrt(np.diag(cov))
    except:
        stderr = np.zeros(n_total)

    event_params = res.x[9:-1]
    event_se = stderr[9:-1]
    
    print("\nEvent Study Coefficients (Level Factor):")
    for i, t in enumerate(model.rel_periods):
        idx = i * 3 # 取 b1 (Level)
        coef = event_params[idx]
        se = event_se[idx]
        
        results_list.append({
            'period': t,
            'coef': coef,
            'se': se
        })
        print(f"t={t:2d}: {coef:.4f} (se: {se:.4f})")
        
    # 加入基准期
    results_list.append({'period': -1, 'coef': 0.0, 'se': 0.0})
    res_df = pd.DataFrame(results_list).sort_values('period')
    
    # 绘图
    plt.figure(figsize=(10, 6))
    plt.errorbar(
        res_df['period'], res_df['coef'], 
        yerr=1.96 * res_df['se'], fmt='o-', capsize=4, 
        color='navy', ecolor='lightblue', label='95% CI'
    )
    plt.axhline(0, color='red', linestyle='--', linewidth=1)
    plt.axvline(-0.5, color='gray', linestyle=':')
    plt.title('Parallel Trend Test (Level Factor)')
    plt.xlabel('Months Relative to Event')
    plt.ylabel('Treatment Effect')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()
    return res_df

def run_placebo_test(
    df: pd.DataFrame,
    true_did_coef: float,
    n_permutations: int = 50,
    fixed_L: float = 0.6
):
    """
    安慰剂检验：随机置换 Treated 状态（按公司聚类）。
    """
    print(f"Running Placebo Test ({n_permutations} permutations)...")
    
    companies = df['CompanyCode'].unique()
    # 真实的 Treated 公司数量
    n_treated = df[df['Treated'] == 1]['CompanyCode'].nunique()
    
    fake_coefs = []
    
    # 预先筛选需要的列以加速拷贝
    # 必须包含: VPYield, Maturity, Post, CompanyCode, EndDt_month
    work_df = df.copy()
    
    for i in range(n_permutations):
        # 1. 随机指定 Treated 公司
        np.random.shuffle(companies)
        fake_treated_firms = set(companies[:n_treated])
        
        # 2. 生成虚假 Treated 和 DiD
        # 注意：这里直接修改内存中的数组比 pandas 操作更快
        is_treated = work_df['CompanyCode'].isin(fake_treated_firms).astype(np.float64)
        is_did = (is_treated * work_df['Post']).astype(np.float64)
        
        # 3. 传入模型（为了速度，我们需要“欺骗”模型类，直接替换其内部数据）
        # 这里为了稳妥，还是实例化新模型，但使用较宽松的收敛条件
        model = DieboldLiFEModel(work_df, max_fe_iters=15, fe_tol=1e-6)
        # 手动覆盖 Treated 和 DiD 数组 (避免重新分配内存)
        model.treated = is_treated.values
        model.did = is_did.values
        
        # 4. 估计
        initial_guess = np.zeros(13)
        initial_guess[-1] = fixed_L
        # 固定 L
        lower = [-np.inf]*12 + [fixed_L - 1e-8]
        upper = [np.inf]*12 + [fixed_L + 1e-8]
        
        try:
            res = least_squares(
                model.residuals, initial_guess, 
                bounds=(lower, upper), max_nfev=50
            )
            # 取 b1Ta (Level 的 DiD 系数)，在参数列表第 9 位 (索引 9)
            # PARAM_ORDER: b1Cb...b3Cb, b1Ca...b3Ca, b1Tb...b3Tb, b1Ta...
            fake_coefs.append(res.x[9])
        except:
            pass
            
        if (i+1) % 10 == 0:
            print(f"Completed {i+1}/{n_permutations}")

    # 绘图
    plt.figure(figsize=(8, 5))
    sns.kdeplot(fake_coefs, fill=True, color='grey', label='Placebo Distribution')
    plt.axvline(true_did_coef, color='red', linestyle='--', label='Actual Estimate')
    plt.title('Placebo Test: Distribution of Placebo DiD Coefficients')
    plt.xlabel('Estimated DiD Coefficient')
    plt.legend()
    plt.show()

In [ ]:
# === 主流程执行 ===

df = prepare_dataframe(INPUT_CSV, KEY_DATE, WINDOW_DAYS)
describe_data(df)

model = DieboldLiFEModel(df, max_fe_iters=MAX_FE_ITERS, fe_tol=FE_TOL)
params, lsq, history = run_nl_regression(
    model,
    initial_L=INITIAL_L,
    max_nl_steps=MAX_NL_STEPS,
    verbose=False,
)
# print_iteration_history(history)

df = df.copy()
df["y_hat"] = model.predict(params)

maturities = [0.25, 0.5, 1, 2, 3, 5, 10, 15]
params_df, stats, cov = summarize_results(model, params, lsq)
print_regression_summary(stats, params_df)
margins = model.margins_did(params, maturities, cov, int(stats["dof"]))
save_outputs(
    df,
    params_df,
    margins,
    output_dir=OUTPUT_DIR,
    scatter_filename=SCATTER_FILENAME,
    export_dta=EXPORT_DTA,
)

summary_payload = {
    "param_order": PARAM_ORDER,
    "params": params.tolist(),
    "optimizer_success": bool(lsq.success),
    "optimizer_message": lsq.message,
    "fe_iterations": model.last_fe_iterations,
    "stats": stats,
}
summary_path = OUTPUT_DIR / "nl_results_summary.json"
summary_path.write_text(json.dumps(summary_payload, indent=2), encoding="utf-8")
summary_payload

In [ ]:
# === 运行平行趋势检验与安慰剂检验 ===

# 1. 获取主模型估计出的 L 值，用于固定参数以加速检验
# 同时也获取主模型中 Level 因子 (b1Ta) 的 DiD 系数，位于索引 9
estimated_L = params[-1]
actual_did_coef = params[9]

logging.info(f"Estimated L: {estimated_L:.4f}, Actual DiD Coef: {actual_did_coef:.4f}")

# 2. 运行平行趋势检验
# window_months 可以根据您的数据窗口调整，例如您用了90天窗口，大约是3个月
pt_results = run_parallel_trend_test(
    df, 
    KEY_DATE, 
    fixed_L=estimated_L, 
    window_months=3
)

# 3. 运行安慰剂检验 (演示仅运行 50 次，建议实际论文中设为 500 次)
run_placebo_test(
    df, 
    true_did_coef=actual_did_coef, 
    n_permutations=50, 
    fixed_L=estimated_L
)